# Prep ParlaSpeech — chapter 1 (variant c)

Parse ParlaSpeech JSONL(s) into canonical pipeline JSONLs, convert per-utterance
FLACs to 16 kHz mono WAVs, and emit **one file per instance-shape** (not per task
— multiple label keys live in one file).

A single run processes **every** ParlaSpeech-{LANG} found under `data/unpacked/`
(or one pinned `cfg.lang`). Each section below defines a function; section 12
loops over the languages and runs them.

**Recipes emitted here** (both whole-utterance, no slicing):
- `utterance_instance` — scalar labels: `speaker_gender`, `filled_pause_present`,
  `filled_pause_count`, `sentiment_logit`, `sentiment_6`. Trainer picks one `label_key`.
- `utterance_frame` — a 50 Hz `filled_pause` label sequence.

Both share the same WAVs and speaker-grouped splits, so nothing leaks and the
splits agree across flavors.

**Not done here** (future recipes, registry stubs below): `event_instance`
(FP-quality, needs the annotator deliverable) and `word_frame` (primary stress,
HR/RS only).

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "1_data_prep":
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp
import utils_audio_splitter as uas

PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

Standard imports.

In [ ]:
import json
from collections import Counter
from dataclasses import dataclass, field, replace

from tqdm.auto import tqdm

---

## 1. Config

- `lang` — `""` → process every ParlaSpeech-{LANG} under `data/unpacked/`; set
  (e.g. `"RS"`) to pin one.
- `recipes` — which instance-shapes to emit.
- `convert_audio` — FLAC → 16 kHz mono WAV. `False` re-writes JSONLs against
  already-converted WAVs without touching audio.
- `num_workers` — parallel FLAC→WAV (each FLAC independent → threads win).
- `audio_index_cache` — persistent `{basename → path}` index, per lang; built
  once, reused after. Delete the file to force a rescan.

`configure(lang)` derives all per-lang paths + the recipe registry; the section-12
driver calls it once per language.

In [ ]:
KNOWN_LANGS = ["HR", "RS", "PL", "CZ"]

@dataclass
class Config:
    lang: str = ""                       # "" → all found; or pin e.g. "RS"
    recipes: tuple = ("utterance_instance", "utterance_frame")

    output_dir:    str = "data/processed_jsonl"
    convert_audio: bool = True
    num_workers:   int  = 8
    cache_index:   bool = True

    frame_rate_hz: int = 50

    split_ratios: tuple = (0.8, 0.1, 0.1)
    split_seed:   str   = "parlaspeech-v1"
    split_group:  str   = "speaker"      # group splits by speaker (no leakage)

    min_duration_s: float = 0.1

    test_mode:      bool = False                                       ############ TEST MODE
    test_n_records: int  = 500

cfg = Config()

# ── Resolve the set of languages to process ───────────────────────────────────
if cfg.lang:
    LANGS = [cfg.lang]
else:
    LANGS = [L for L in KNOWN_LANGS
             if (PROJECT_ROOT / f"data/unpacked/ParlaSpeech-{L}").exists()]
    if not LANGS:
        raise FileNotFoundError(
            "cfg.lang empty and no ParlaSpeech-{LANG} dirs under data/unpacked/. "
            "Run 10_download_data.ipynb first.")

if cfg.test_mode:
    udp.banner("🧪 TEST MODE", char="-")
print(f"languages to process: {LANGS}")

def configure(lang: str) -> dict:
    """Per-lang view: paths + recipe registry. (build_recipes defined in §2.)"""
    L, low = lang, lang.lower()
    DATASET = f"ParlaSpeech-{L}"
    OUT_DIR = cfg.output_dir if not cfg.test_mode else "data/test_processed_jsonl"
    WAV_DIR = (f"data/cut_audio/ParlaSpeech-{L}" if not cfg.test_mode
               else f"data/cut_audio/test/ParlaSpeech-{L}")
    ctx = dict(
        L=L, l=low, DATASET=DATASET, OUT_DIR=OUT_DIR, WAV_DIR=WAV_DIR,
        jsonl_path=f"data/unpacked/ParlaSpeech-{L}/ParlaSpeech-{L}.v3.0/ParlaSpeech-{L}.v3.0.jsonl",
        audio_base_dir=f"data/unpacked/ParlaSpeech-{L}-audio",
        audio_index_cache=f"data/processed_jsonl/parlaspeech_{low}_audio_index.json",
        RECIPES=build_recipes(OUT_DIR, low),
    )
    for name in cfg.recipes:
        if name not in ctx["RECIPES"]:
            raise ValueError(f"unknown recipe {name!r}. Known: {sorted(ctx['RECIPES'])}")
    return ctx

---

## 2. Recipe registry

A *recipe* is an instance-shape: `(unit × cut × instance|frame)`. Multiple label
keys can live in one recipe — task type is a downstream parameter, not a reason
to split files. This mirrors the `TARGETS` dict in `30_train_instance.ipynb`.

`requires` gates corpus-specific tiers (e.g. `primary_stress` is HR/RS only) so a
recipe fails loudly on a corpus that lacks them rather than emitting garbage.

In [ ]:
def build_recipes(OUT_DIR: str, l: str) -> dict:
    return {
        # ---- emitted here (whole utterance, no cut) -------------------------
        "utterance_instance": dict(
            unit="utterance", level="instance", cut=False, requires=(),
            out=f"{OUT_DIR}/parlaspeech_{l}_utterance_instance.jsonl"),
        "utterance_frame": dict(
            unit="utterance", level="frame", cut=False, requires=("filled_pauses",),
            out=f"{OUT_DIR}/parlaspeech_{l}_utterance_frame.jsonl"),

        # ---- future (need cut=True via utils_audio_splitter) ----------------
        # "event_instance": one record per FP event; label = FP quality
        #   (vowel / vowel+nasal / nasal / other / NA). Needs annotator. unit="event".
        # "word_frame": one record per word; 50 Hz primary-stress sequence.
        #   HR/RS only. unit="word", requires=("primary_stress","words_align").
    }

---

## 3. Locate the JSONL

In [ ]:
def locate_jsonl(ctx: dict) -> Path:
    p = PROJECT_ROOT / ctx["jsonl_path"]
    if not p.exists():
        raise FileNotFoundError(
            f"JSONL not found: {p}\nRun 10_download_data.ipynb for {ctx['DATASET']} first.")
    print(f"✅ {p.relative_to(PROJECT_ROOT)}  ({p.stat().st_size/1e6:.0f} MB)")
    return p

---

## 4. Preflight — peek at one record

Confirms which tiers this language actually has (sentiment/words/`_align` vary).

In [ ]:
def preflight(jsonl_path: Path) -> None:
    with open(jsonl_path, encoding="utf-8") as f:
        ex = json.loads(f.readline())
    si = ex.get("speaker_info", {})
    fps = ex.get("filled_pauses")
    fp_state = "failed" if fps is None else ("none" if fps == [] else f"{len(fps)} FP")
    print(f"  e.g. {ex['id']}  | {si.get('Speaker_gender','?')} / {si.get('Lang','?')} "
          f"| FP={fp_state} | sentiment={'sentiment' in ex} "
          f"| words={'words' in ex} | aligns={'words_align' in ex}")

---

## 5. Parse — build canonical records

One canonical record per utterance, carrying **every** label key. `None` means
"not available for this utterance" (FP inference failed, or gender `"-"`); the
trainer drops `None` per-task, so one failed tier never costs the others.

Keeps the `words` tier (needed for future word-level recipes). The two `_align`
tiers are large and HR/RS-only — extraction lines present but commented out.

In [ ]:
def gender_label(si):
    g = si.get("Speaker_gender")
    return g if g in ("M", "F") else None   # "-"/missing → None (dropped per-task)

def parse_records(ctx: dict, jsonl_path: Path) -> list[dict]:
    DATASET, WAV_DIR = ctx["DATASET"], ctx["WAV_DIR"]

    def parse_record(r):
        dur = round(float(r.get("audio_length", 0.0)), 3)
        if dur < cfg.min_duration_s:
            return None
        si  = r.get("speaker_info", {})
        fps = r.get("filled_pauses")                 # None = failed; [] = none
        sent = r.get("sentiment") or {}
        raw_audio = r.get("audio")
        file_hash = Path(raw_audio).parts[0] if raw_audio else None
        stem      = Path(raw_audio).stem if raw_audio else r["id"]
        return {
            "instance_id": r["id"],
            "dataset":     DATASET,
            "file_id":     file_hash,
            "audio_path":  f"{WAV_DIR}/{file_hash}/{stem}.wav",
            "speaker":     si.get("Speaker_ID", "unknown"),
            "text":        r.get("text"),
            "labels": {
                "speaker_gender":       gender_label(si),
                "filled_pause_present": None if fps is None else int(bool(fps)),
                "filled_pause_count":   None if fps is None else len(fps),
                "sentiment_logit":      sent.get("ParlaSent_logit"),
                "sentiment_6":          sent.get("ParlaSent_6"),
            },
            "metadata": {
                "source_audio": raw_audio,
                "audio_length": dur,
                "lang":         si.get("Lang"),
                "speaker_info": si,
                "words":        r.get("words"),
                # "words_align": r.get("words_align"),   # HR/RS only, bulky
                # "chars_align": r.get("chars_align"),   # HR/RS only, bulky
                "filled_pauses": fps,                    # scratch (for frame recipe; stripped on write)
            },
        }

    records, n_total, n_short = [], 0, 0
    with open(jsonl_path, encoding="utf-8") as f:
        for line in tqdm(f, desc=f"parsing {DATASET}", unit=" lines", leave=False):
            n_total += 1
            if cfg.test_mode and n_total > cfg.test_n_records:
                break
            rec = parse_record(json.loads(line))
            if rec is None:
                n_short += 1
                continue
            records.append(rec)
    print(f"  parsed {n_total} lines  kept {len(records)}  dropped(short) {n_short}")
    return records

---

## 6. Stats

In [ ]:
def print_stats(records: list[dict]) -> None:
    gender = Counter(r["labels"]["speaker_gender"] for r in records)
    fp_known = [r for r in records if r["labels"]["filled_pause_present"] is not None]
    n_fp_pos = sum(r["labels"]["filled_pause_present"] for r in fp_known)
    sent_known = sum(1 for r in records if r["labels"]["sentiment_logit"] is not None)
    speakers = {r["speaker"] for r in records}
    print(f"  speakers: {len(speakers)}  gender: {dict(gender)}")
    if fp_known:
        print(f"  FP labelled: {len(fp_known)} (failed {len(records)-len(fp_known)})  "
              f"present: {n_fp_pos} ({100*n_fp_pos/len(fp_known):.1f}%)")
    print(f"  sentiment logits: {sent_known}")

---

## 7. Assign splits — grouped by speaker

Deterministic; the same speaker always lands in the same split, so no speaker
leaks between train/dev/test.

In [ ]:
def do_splits(records: list[dict]) -> None:
    udp.assign_splits(records, ratios=cfg.split_ratios,
                      group_key=cfg.split_group, seed=cfg.split_seed, overwrite=True)
    print(f"  splits: {udp.split_summary(records)}")

---

## 8. Convert FLAC → WAV

Whole-file convert via `utils_audio_splitter`. The basename-index resolver finds
each FLAC regardless of `partX/` nesting; the index is cached so re-runs skip the
scan. Unresolved records are dropped. Prints where the cut audio landed; per-lang
stats only when something was missing/failed/skipped.

In [ ]:
def convert_audio(ctx: dict, records: list[dict]) -> list[dict]:
    WAV_DIR = ctx["WAV_DIR"]
    if not cfg.convert_audio:
        print(f"  convert_audio=False — assuming WAVs under {WAV_DIR}")
        return records
    resolver = uas.make_flac_index_resolver(
        ctx["audio_base_dir"],
        record_key_path=("metadata", "source_audio"),
        index_cache_path=(ctx["audio_index_cache"] if cfg.cache_index else None),
    )
    records, stats = uas.cut_dataset(records, resolver, num_workers=cfg.num_workers)
    msg = f"🔊 cut audio → {WAV_DIR}  (kept {stats['kept']})"
    extra = [f"{k} {stats[k]}" for k in ("missing_source", "cut_failed", "skipped_existing")
             if stats[k]]
    if extra:
        msg += "  [" + ", ".join(extra) + "]"
    print(msg)
    return records

---

## 9. Frame label helper

50 Hz binary sequence from the `filled_pauses` intervals.

In [ ]:
def compute_frame_labels(filled_pauses, dur, hz):
    n = round(dur * hz)
    labels = [0] * n
    for fp in (filled_pauses or []):
        s = max(0, round(fp["time_s"] * hz))
        e = min(n, round(fp["time_e"] * hz))
        for i in range(s, e):
            labels[i] = 1
    return labels

---

## 10. Write recipe JSONLs

`utterance_instance` carries the scalar labels. `utterance_frame` carries the
50 Hz `filled_pause` sequence and only includes utterances where FP inference
succeeded. The raw `filled_pauses` scratch field is stripped from both outputs.

In [ ]:
def build_instance(r):
    out = {k: r[k] for k in ("instance_id", "dataset", "file_id",
                             "audio_path", "speaker", "text", "split")}
    out["labels"]   = dict(r["labels"])
    out["metadata"] = {k: v for k, v in r["metadata"].items() if k != "filled_pauses"}
    return out

def build_frame(r):
    fps = r["metadata"]["filled_pauses"]
    dur = r["metadata"]["audio_length"]
    out = {k: r[k] for k in ("instance_id", "dataset", "file_id",
                             "audio_path", "speaker", "text", "split")}
    out["frame_rate_hz"] = cfg.frame_rate_hz
    out["labels"] = {"filled_pause": compute_frame_labels(fps, dur, cfg.frame_rate_hz)}
    out["metadata"] = {k: v for k, v in r["metadata"].items() if k != "filled_pauses"}
    return out

def write_recipes(ctx: dict, records: list[dict]) -> dict:
    written = {}
    for name in cfg.recipes:
        spec = ctx["RECIPES"][name]
        if spec["level"] == "instance":
            rows = [build_instance(r) for r in records]
        else:
            rows = [build_frame(r) for r in records
                    if r["metadata"]["filled_pauses"] is not None]
        n = udp.write_jsonl(rows, spec["out"])
        written[name] = (spec["out"], n)
        print(f"  ✅ {name}: {n} → {spec['out']}")
    return written

---

## 11. Sanity checks

In [ ]:
def sanity(ctx: dict, written: dict) -> None:
    for name, (path, n) in written.items():
        rows = udp.read_jsonl(path)
        n_tot, n_valid, errs = udp.validate_jsonl(rows)
        tag = "✅" if not errs else "⚠️ "
        print(f"  {tag} {name}: {n_valid}/{n_tot} valid")
        for e in errs[:3]:
            print(f"       {e}")
        if ctx["RECIPES"][name]["level"] == "frame":
            bad = sum(1 for r in rows
                      if abs(len(r["labels"]["filled_pause"])
                             - round(r["metadata"]["audio_length"] * cfg.frame_rate_hz)) > 1)
            print(f"       frame-length vs duration: {bad} mismatches (>1 frame)")
        else:
            present = Counter(r["labels"]["filled_pause_present"] for r in rows)
            null_wav = sum(1 for r in rows if not r.get("audio_path"))
            print(f"       filled_pause_present: {dict(present)}  | audio_path null: {null_wav}")

---

## 12. Run — all languages

One pass per language; each is parsed, split, converted, and written before the
next, so records don't accumulate across languages.

In [ ]:
summary = {}
for LANG in LANGS:
    udp.banner(f"▶  {LANG}", char="=")
    ctx = configure(LANG)
    jsonl_path = locate_jsonl(ctx)
    preflight(jsonl_path)
    records = parse_records(ctx, jsonl_path)
    print_stats(records)
    do_splits(records)
    records = convert_audio(ctx, records)
    written = write_recipes(ctx, records)
    sanity(ctx, written)
    summary[LANG] = {name: n for name, (p, n) in written.items()}
    del records                      # free before the next language

udp.banner("ALL LANGUAGES DONE", char="=")
for lng, w in summary.items():
    print(f"  {lng}: " + ", ".join(f"{k}={v}" for k, v in w.items()))

---

## Next

- **Chapter 2** — `20_sniff_dataset.ipynb` at either JSONL.
- **Chapter 3** — add a `TARGETS` entry: `parlaspeech_{lang}_utterance_instance.jsonl`,
  `label_key` ∈ `{speaker_gender, filled_pause_present, filled_pause_count}`
  (classification) or `sentiment_logit` (regression).
- **Chapter 4** — frame trainer at `..._utterance_frame.jsonl`.